# Chapter 10 — Rough, Temporal, and Fuzzy Modelling
### Notebook 1 · Time: Allen's interval algebra

*Book reference: Section 10.1*

Two intervals can stand in exactly thirteen relations. That fact, plus a composition table, is enough to catch temporal contradictions no human reviewer would notice.

In [ ]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

In [ ]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch10_toolkit as ch10
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

## 1. The thirteen relations

Jointly exhaustive and pairwise disjoint: between any two intervals **exactly one** holds. Six come in inverse pairs, plus `equals`.

In [ ]:
rows = [{'code': code, 'name': name,
         'inverse': ch10.inverse(code)} for code, name in ch10.ALLEN_NAMES.items()]
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
examples = [((0, 2), (3, 5)), ((0, 3), (3, 5)), ((0, 4), (2, 6)),
            ((1, 3), (1, 5)), ((2, 3), (1, 5)), ((3, 5), (1, 5)),
            ((1, 4), (1, 4))]
for a, b in examples:
    r = ch10.relation_between(a, b)
    print(f'  A={a}  B={b}   A {ch10.ALLEN_NAMES[r]:14s} B   ({r})')

### Verifying 'exactly one'

The claim is checkable, so let us check it rather than repeat it.

In [ ]:
intervals = [(s, e) for s in range(5) for e in range(s + 1, 6)]
seen = {}
for a in intervals:
    for b in intervals:
        seen.setdefault(ch10.relation_between(a, b), 0)
        seen[ch10.relation_between(a, b)] += 1
print(f'{len(seen)} distinct relations over {len(intervals)**2} interval pairs')
print(dict(sorted(seen.items())))
assert len(seen) == 13

## 2. The composition table, derived

If `A r₁ B` and `B r₂ C`, which relations can hold between A and C? Allen published the 13×13 answer; we **compute** it by enumerating concrete intervals and recording what actually occurs. A table with 169 entries is exactly the kind of thing worth deriving rather than transcribing.

In [ ]:
for first, second in [('b', 'b'), ('m', 'm'), ('d', 'b'), ('o', 'o'), ('s', 'f')]:
    result = sorted(ch10.compose(first, second))
    print(f'  {ch10.ALLEN_NAMES[first]:14s} o {ch10.ALLEN_NAMES[second]:14s} '
          f'-> {result}')

In [ ]:
sizes = {}
for (first, second), result in ch10.composition_table().items():
    sizes[len(result)] = sizes.get(len(result), 0) + 1
print('how informative is a composition?')
for size, count in sorted(sizes.items()):
    print(f'  {count:3d} entries yield {size:2d} possible relation(s)')
print('\nSome compositions pin the answer exactly (before o before = before);\n'
      'others barely constrain it at all. That spread is why temporal\n'
      'reasoning is search rather than lookup.')

## 3. Catching a contradiction

`A before B`, `B before C`, `C before A`. Obvious on three intervals; not obvious at all on thirty. Path consistency composes through every third interval until a label set empties.

In [ ]:
network = ch10.Network.complete(('A', 'B', 'C'),
                                {('A', 'B'): {'b'}, ('B', 'C'): {'b'},
                                 ('A', 'C'): {'bi'}})
print('constraints: A before B, B before C, A after C')
print('result     :', ch10.path_consistent(network))
print('\nThe A-C label emptied: no assignment of intervals can satisfy all three.')

In [ ]:
network = ch10.Network.complete(('A', 'B', 'C'),
                                {('A', 'B'): {'b'}, ('B', 'C'): {'b'}})
result = ch10.path_consistent(network)
print('constraints: A before B, B before C  (A-C left open)')
print('result     :', result)
print('inferred A-C:', sorted(network.get('A', 'C')))
print('\nPropagation did not just check -- it INFERRED. A before C follows,\n'
      'and the label narrowed from thirteen possibilities to one.')

> **Sound but incomplete.** Path consistency can *prove* inconsistency, but a path-consistent network is not guaranteed satisfiable — deciding that in general is NP-complete. This is the same expressivity/decidability bargain as Chapters 2 and 3, in temporal clothing, and it is why Notebook 4 prices temporal reasoning as **high**.

### Exercise 1.1 — Find the inconsistency in a five-interval schedule

Build a network over five intervals containing a cycle, and confirm the checker finds it. Then remove one constraint and confirm it becomes consistent.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution 1.1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
constraints = {('A', 'B'): {'b'}, ('B', 'C'): {'b'}, ('C', 'D'): {'b'},
               ('D', 'E'): {'b'}, ('E', 'A'): {'b'}}
network = ch10.Network.complete(('A', 'B', 'C', 'D', 'E'), constraints)
result = ch10.path_consistent(network)
print('with the cycle E before A:', result)
assert not result['consistent']

relaxed = dict(constraints); relaxed.pop(('E', 'A'))
network2 = ch10.Network.complete(('A', 'B', 'C', 'D', 'E'), relaxed)
result2 = ch10.path_consistent(network2)
print('without it              :', result2)
print('inferred A-E            :', sorted(network2.get('A', 'E')))
assert result2['consistent']
print('\nFive intervals is already past the point where a human reviewer would\n'
      'reliably spot the cycle by eye, and real schedules have hundreds.')

### Exercise 1.2 — Check a composition against the published table

Allen's table says `during ∘ before = before`. Verify it from the derived table, and find a composition that yields the **most** possible relations.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution 1.2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
assert sorted(ch10.compose('d', 'b')) == ['b']
print('during o before ->', sorted(ch10.compose('d', 'b')), ' (matches Allen)')

table = ch10.composition_table()
worst = max(table.items(), key=lambda kv: len(kv[1]))
(first, second), result = worst
print(f'\nleast informative: {ch10.ALLEN_NAMES[first]} o {ch10.ALLEN_NAMES[second]}'
      f' -> {len(result)} of 13 relations')
print('  ', sorted(result))
assert len(result) >= 9
print('\nSome compositions tell you almost nothing. A solver that propagates\n'
      'those first wastes its budget -- which is exactly what the MDP in\n'
      'Notebook 4 optimises.')